# A2.9 · The classic failures

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.8 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**.

| | |
|---|---|
| Open-source tooling | SPIRE, Keycloak |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The classic identity failures did not go away when the caller became an agent.
They got faster, harder to attribute, and in one case genuinely worse.

This lesson runs the whole track's controls against a suite of them, so you can
see which are closed by what you have built and which are not:

| Failure | Age | What changed with agents |
|---|---|---|
| Credential in source | ancient | Agents read source, so a leaked key is now *actionable* by the reader |
| Over-broad scope | ancient | Granted programmatically, at machine speed, with no reviewer |
| Replay of a stolen token | ancient | Same, but the thief acts in milliseconds |
| Confused deputy | 1988 | The agent *is* a deputy, by design |
| Privilege escalation via chain | ancient | Chains are now 3–5 hops and nobody drew them |
| Impersonation | ancient | **Worse**: now the default deployment pattern |

The honest outcome of this lesson is that the controls built in A2.1–A2.8 close
most of these, and one of them — impersonation — cannot be closed by a token
format at all. It requires a platform decision: agents must not be *issuable* a
principal's credential in the first place.

## 2 · Demo — the delegation implementation from A2.5, under attack

Rebuild the minimum needed, then fire the suite at it.

In [ ]:
import time
from dataclasses import dataclass, field

CEILINGS = {"dana@corp": {"repo:read", "repo:write", "deploy:prod", "secrets:read"},
            "patch-agent": {"repo:read", "repo:write"},
            "triage-agent": {"repo:read"},
            "deploy-agent": {"repo:read", "deploy:prod"}}

class DelegationError(Exception): pass

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    issued: float = field(default_factory=time.time); ttl: float = 300
    @property
    def expired(self): return time.time() - self.issued > self.ttl
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

def mint(p, s=None):
    want = set(s) if s else set(CEILINGS[p])
    if not want <= CEILINGS[p]: raise DelegationError("over ceiling")
    return Token(p, p, want)

def exchange(pres, actor, scopes):
    if pres.expired: raise DelegationError("presented token expired")
    scopes = set(scopes)
    if not scopes <= pres.scopes:
        raise DelegationError(f"not in presented token: {sorted(scopes - pres.scopes)}")
    if not scopes <= CEILINGS.get(actor, set()):
        raise DelegationError(f"above {actor}'s ceiling: "
                              f"{sorted(scopes - CEILINGS.get(actor,set()))}")
    return Token(pres.sub, actor, scopes, {"actor": pres.actor, "act": pres.act})

def impersonate(p, actor, scopes):
    return Token(p, p, set(scopes), None)

dana  = mint("dana@corp")
patch = exchange(dana, "patch-agent", {"repo:read", "repo:write"})
print("baseline chain:", " → ".join(patch.chain()))

In [ ]:
SUITE = [
 ("IDN-01", "escalate scope during delegation", "critical",
  lambda: exchange(patch, "deploy-agent", {"deploy:prod"})),
 ("IDN-02", "exceed the recipient's ceiling", "high",
  lambda: exchange(dana, "triage-agent", {"repo:write"})),
 ("IDN-03", "replay an expired token", "high",
  lambda: exchange(Token("dana@corp", "patch-agent", {"repo:write"}, ttl=-1),
                   "deploy-agent", {"repo:read"})),
 ("IDN-04", "confused deputy: reuse the chain for an unrelated task", "high",
  lambda: exchange(patch, "patch-agent", {"repo:write"})),
]
results = []
for aid, name, sev, fn in SUITE:
    try:
        fn(); got_through, detail = True, "succeeded"
    except DelegationError as e:
        got_through, detail = False, str(e)[:48]
    results.append((aid, name, sev, got_through, detail))
    print(f"{aid}  {'GOT THROUGH' if got_through else 'blocked    '}  {name}")
    print(f"        {detail}")

# impersonation is not a token-format failure; it is a platform one
bad = impersonate("dana@corp", "patch-agent", {"repo:write"})
hidden = "patch-agent" not in bad.chain()
results.append(("IDN-05", "impersonation (agent uses the human's credential)",
                "critical", hidden, "agent absent from the chain"))
print(f"IDN-05  {'GOT THROUGH' if hidden else 'blocked    '}  "
      f"impersonation — chain is {bad.chain()}")

## 3 · Where it breaks — read the one that got through

Four of five are closed by the narrowing rules. IDN-05 succeeds, and it succeeds **by design**: nothing inside a token format can stop a caller choosing not to use delegation at all. If an agent can obtain a principal's credential, it can always present it.

Note also IDN-04 — the confused deputy. It is *blocked here* only because the scope re-request is a no-op; a genuine confused-deputy attack works at the content layer, not the token layer, and belongs to the injection surface (C1.3).

In [ ]:
asr = sum(1 for *_, got, _ in results if got) / len(results)
print(f"attack success rate against the identity surface: {asr:.0%}")
print("\nby severity:")
for aid, name, sev, got, detail in results:
    if got:
        print(f"   {sev.upper():9s} {aid} — {name}")

## 4 · The control for the one that got through

Impersonation is closed at the platform, not the protocol. Three mechanisms, in descending order of strength.

In [ ]:
CONTROLS = [
 ("issuance", "The IdP refuses to issue a human-subject credential to a workload "
              "identity. An agent literally cannot obtain Dana's token.",
              "strongest — removes the capability", True),
 ("binding",  "Tokens are sender-constrained (mTLS / DPoP), so a token minted for "
              "Dana's browser cannot be presented by the agent's workload.",
              "strong — token is useless off its holder", True),
 ("detection","Alert when a human-subject token is presented from a workload "
              "network identity or at machine rate.",
              "weakest — after the fact, but deployable this week", False),
]
for name, how, strength, preventive in CONTROLS:
    print(f"{name:10s} [{'preventive' if preventive else 'detective':10s}] {strength}")
    print(f"           {how}\n")

def token_binding_check(token, presenter_workload, bound_to):
    """Sender-constrained tokens: presenting from the wrong workload fails."""
    if bound_to and presenter_workload != bound_to:
        return False, (f"token bound to {bound_to}, presented by "
                       f"{presenter_workload}")
    return True, "binding ok"

print("verify — the same impersonation attempt, with sender-constrained tokens:")
for workload in ("dana-browser-session", "spiffe://corp/ns/prod/sa/patch-agent"):
    ok, why = token_binding_check(bad, workload, bound_to="dana-browser-session")
    print(f"   presented by {workload:42s} {'ACCEPTED' if ok else 'REJECTED'} — {why}")

## What you just proved

IDN-01 through IDN-04 are all blocked by the narrowing rules, each naming the rule that refused it. IDN-05 (impersonation) gets through, giving an identity-surface attack success rate of 20%, and the chain for the impersonated token contains only `dana@corp`. Sender-constrained binding then rejects the same token when presented from the agent's workload identity.

## Your turn

IDN-05 is the one that matters and the one your platform probably allows. Find out whether your IdP will issue a human-subject token to a workload. If it will, write the detection first — it ships in a week — and open the issuance change as the real fix.

---

**Next → [A3.1 · Sandboxing is the perimeter](https://spbreed.github.io/cyber-commons/lessons/A3.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*